<h3>Batch processing notebook based on 3D Cellpose SP logic (no napari visualization)</h3>

In [ ]:
from pathlib import Path
from tqdm import tqdm
import czifile
import numpy as np
import tifffile
import pandas as pd
from skimage.measure import regionprops_table
import pyclesperanto_prototype as cle

from utils import (
    extract_scaling_metadata,
    predict_nuclei_labels,
    simulate_cytoplasm,
    segment_organoids_from_cp_labels,
)
from data_analysis_utils import extract_organoid_stats_and_merge
from io_utils import (
    list_images,
    ensure_output_dir,
    load_precomputed_results_if_available,
)

cle.select_device("RTX")

### Configure experiment inputs
Set your input directory and analysis parameters.

In [ ]:
# Path where .czi images are stored
RAW_DATA_DIRECTORY = "./raw_data/WT_organoids/"
#RAW_DATA_DIRECTORY = "./raw_data/20240703 PGE2MSN Tum_organoids"

# Channel index used for CellposeSAM-based 3D nuclei segmentation
NUCLEI_CHANNEL = 2

# YAP intensity channel used for nuclei/cytoplasm measurements
YAP_CHANNEL = 1

# Cytoplasm thickness in voxels (starting from the nucleus)
# If padding is applied, the thickness will be applied to the padded nucleus
CYTOPLASM_THICKNESS = 2

# Padding to keep a safe distance between nuclei and cytoplasm borders
# It expands the nucleus by the padding value in all directions
NUCLEI_PADDING = 0

directory_path = Path(RAW_DATA_DIRECTORY)
input_folder_id = directory_path.stem
images = list_images(directory_path, "czi")
len(images)

### Run Batch Analysis

In [ ]:
# Prepare output directories
nuclei_labels_dir = ensure_output_dir(RAW_DATA_DIRECTORY, input_folder_id, results_type="nuclei_labels")
bp_results_dir = ensure_output_dir("./results", input_folder_id, results_type="bp_results")

dataframes = []

for image in tqdm(images):
    # Read path storing raw image and extract filename
    file_path = Path(image)
    filename = file_path.stem

    # Read the image file and remove singleton dimensions
    img = czifile.imread(image)
    img = img.squeeze()

    # Extract experiment, mouse, treatment and replica ids
    experiment_id = filename.split(" ")[0]
    mouse_id = filename.split(" ")[1]
    treatment_id = filename.split(" ")[2]
    replica_id = filename.split(" ")[-1]

    # Calculate anisotropy correction factor for Cellpose 3D inference
    scaling_x_um, scaling_y_um, scaling_z_um = extract_scaling_metadata(file_path)
    rescale_factor = scaling_z_um / np.mean([scaling_x_um, scaling_y_um])

    nuclei_labels_path = nuclei_labels_dir / f"{filename}_nuclei_labels.tif"

    # Load existing labels, otherwise predict and save them
    nuclei_labels = load_precomputed_results_if_available(
        nuclei_labels_dir, filename, results_type="nuclei_labels"
    )

    if nuclei_labels is None:
        nuclei_labels = predict_nuclei_labels(
            img,
            rescale_factor,
            NUCLEI_CHANNEL,
            min_max_nuclei_volume=None,
            visualize=False,
        )
        tifffile.imwrite(nuclei_labels_path, nuclei_labels)

    # Simulate cytoplasm and measure; if label/YAP shapes mismatch (stale cache), re-segment and overwrite
    _shape_err = "Label and intensity image shapes must match"
    for _attempt in range(2):
        cytoplasm = simulate_cytoplasm(nuclei_labels, cytoplasm_thickness=CYTOPLASM_THICKNESS, nuclei_padding=NUCLEI_PADDING)
        # Use YAP channel for intensity-based measurements
        try:
            props_nuclei = regionprops_table(
                label_image=nuclei_labels,
                intensity_image=img[YAP_CHANNEL],
                properties=["label", "intensity_mean", "area"],
            )
            props_cytoplasm = regionprops_table(
                label_image=cytoplasm,
                intensity_image=img[YAP_CHANNEL],
                properties=["label", "intensity_mean", "area"],
            )
            break
        except ValueError as e:
            if _attempt == 0 and _shape_err in str(e):
                nuclei_labels = predict_nuclei_labels(
                    img,
                    rescale_factor,
                    NUCLEI_CHANNEL,
                    min_max_nuclei_volume=None,
                    visualize=False,
                )
                tifffile.imwrite(nuclei_labels_path, nuclei_labels)
                continue
            raise

    # Segment organoids using Cellpose generated labels as seeds
    organoid_labels = segment_organoids_from_cp_labels(nuclei_labels, cytoplasm_thickness=1)

    # Build and merge per-label dataframes
    df_nuclei = pd.DataFrame(props_nuclei)
    df_cytoplasm = pd.DataFrame(props_cytoplasm)

    df_nuclei.rename(columns={"intensity_mean": "intensity_mean_nuclei", "area": "volume_nuclei"}, inplace=True)
    df_cytoplasm.rename(columns={"intensity_mean": "intensity_mean_cyto", "area": "volume_cyto"}, inplace=True)

    merged_df = pd.merge(df_nuclei, df_cytoplasm, on="label")

    # Calculate nuclei/cytoplasm ratio of mean YAP intensity
    merged_df["nuclei_cyto_ratio"] = merged_df["intensity_mean_nuclei"] / merged_df["intensity_mean_cyto"]

    # Extract organoid stats and map nuclei_labels to parent organoid_labels
    # Nuclei labels not mapped to an organoid get an organoid_id mapping of 0 value
    merged_df = extract_organoid_stats_and_merge(nuclei_labels, organoid_labels, merged_df)

    # Add image metadata ids
    merged_df.insert(0, "experiment_id", experiment_id)
    merged_df.insert(1, "mouse_id", mouse_id)
    merged_df.insert(2, "treatment_id", treatment_id)
    merged_df.insert(3, "replica_id", replica_id)
    merged_df.insert(4, "min_nuc_vol", None)
    merged_df.insert(5, "max_nuc_vol", None)
    merged_df.insert(6, "cyto_thickness", CYTOPLASM_THICKNESS)
    merged_df.insert(7, "nuclei_padding", NUCLEI_PADDING)

    dataframes.append(merged_df)

# Concatenate per-image dataframes
final_df = pd.concat(dataframes, ignore_index=True)

# Save per-label results
per_label_path = bp_results_dir / f"per_label_results_{input_folder_id}_MIN_None_MAX_None_CT{CYTOPLASM_THICKNESS}_NP{NUCLEI_PADDING}.csv"
final_df.to_csv(per_label_path, index=False)

# Display first 10 rows
final_df.head(10)